<div align="center">

# MegaFlow: Zero-Shot Large Displacement Optical Flow

**[Dingxi Zhang](https://kristen-z.github.io/)** · **[Fangjinhua Wang](https://fangjinhuawang.github.io/)** · **[Marc Pollefeys](https://people.inf.ethz.ch/marc.pollefeys/)** · **[Haofei Xu](https://haofeixu.github.io/)**

*ETH Zurich · Microsoft · University of Tübingen, Tübingen AI Center*

[![Project Page](https://img.shields.io/badge/Project-Page-blue?style=flat&logo=Google%20chrome&logoColor=white)](https://kristen-z.github.io/projects/megaflow/)
[![arXiv](https://img.shields.io/badge/arXiv-Paper-b31b1b.svg?style=flat&logo=arxiv&logoColor=white)](https://arxiv.org/abs/2603.25739)
[![HuggingFace](https://img.shields.io/badge/🤗%20HuggingFace-Models-yellow.svg)](https://huggingface.co/Kristen-Z/MegaFlow)
[![GitHub](https://img.shields.io/badge/GitHub-Code-black?style=flat&logo=github)](https://github.com/cvg/megaflow)

</div>

---

This notebook demonstrates **MegaFlow** for:
1. 🌈 **Optical flow estimation** — dense flow between consecutive video frames
2. 🎯 **Point tracking** — long-range tracking of a grid of points across a video

Pretrained models are auto-downloaded from HuggingFace. A GPU runtime is strongly recommended.

## 1. Installation

In [ ]:
# Install MegaFlow and its dependencies
!pip install -q git+https://github.com/cvg/megaflow.git

In [ ]:
import torch
import cv2
import numpy as np
import torch.nn.functional as F
import imageio
import os
from IPython.display import Video, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download Example Videos

We use example videos from the MegaFlow repository. You can also upload your own video below.

In [ ]:
# Download example assets
BASE_URL = "https://raw.githubusercontent.com/cvg/megaflow/main/assets"
os.makedirs("assets", exist_ok=True)

for fname in ["apple.mp4", "longboard.mp4", "chamaleon.mp4"]:
    if not os.path.exists(f"assets/{fname}"):
        !wget -q -O assets/{fname} {BASE_URL}/{fname}
        print(f"Downloaded assets/{fname}")

print("Example videos ready.")

In [ ]:
# --- Optional: upload your own video instead ---
# from google.colab import files
# uploaded = files.upload()
# INPUT_VIDEO = list(uploaded.keys())[0]

INPUT_VIDEO = "assets/longboard.mp4"  # change to any of: apple.mp4, chamaleon.mp4
print(f"Input video: {INPUT_VIDEO}")

## 3. Shared Utilities

In [ ]:
def calculate_dynamic_size_flow(orig_h, orig_w, patch_size=14):
    """Resize longest edge to 952, keep aspect ratio, align to patch_size."""
    fix_width = 952
    if orig_w >= orig_h:
        new_w = fix_width
        new_h = round(orig_h * (new_w / orig_w) / patch_size) * patch_size
    else:
        new_h = fix_width
        new_w = round(orig_w * (new_h / orig_h) / patch_size) * patch_size
    return int(new_h), int(new_w)


def calculate_dynamic_size_track(orig_h, orig_w, patch_size=14):
    """Fix width to 518, preserve aspect ratio, align height to patch_size."""
    new_w = 518
    new_h = round(orig_h * (new_w / orig_w) / patch_size) * patch_size
    return int(new_h), int(new_w)


def load_video(path, mode="flow"):
    """Load a video into a list of RGB numpy frames, resized for the given mode."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 24.0

    frames, orig_shape = [], None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if orig_shape is None:
            orig_shape = frame.shape[:2]
        if mode == "flow":
            new_h, new_w = calculate_dynamic_size_flow(orig_shape[0], orig_shape[1])
        else:
            new_h, new_w = calculate_dynamic_size_track(orig_shape[0], orig_shape[1])
        frame = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        frames.append(frame)
    cap.release()
    return frames, orig_shape, fps

---
## 4. 🌈 Optical Flow Estimation

MegaFlow estimates dense optical flow between each pair of consecutive frames.  
Results are colour-coded using the standard Middlebury colour wheel.

In [ ]:
from megaflow.model import MegaFlow
from megaflow.utils.flow_viz import flow_to_image

print("Loading megaflow-flow from HuggingFace (auto-cached after first download)...")
flow_model = MegaFlow.from_pretrained("megaflow-flow", device=device)
flow_model.eval()
print("Model ready.")

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
FLOW_WINDOW_SIZE = 4     # sliding-window size (frames processed together)
FLOW_ITERS       = 8     # refinement iterations
FLOW_RESTORE     = True  # upsample flow back to original resolution
FLOW_OUTPUT      = "output_flow.mp4"
# ────────────────────────────────────────────────────────────────────────────

frames_np, native_size, fps = load_video(INPUT_VIDEO, mode="flow")
print(f"Loaded {len(frames_np)} frames  |  processing size: {frames_np[0].shape[:2]}  |  native: {native_size}  |  fps: {fps:.1f}")

input_image = [torch.from_numpy(f).permute(2, 0, 1).float() for f in frames_np]
input_scene = torch.stack(input_image, dim=0)[None]  # (1, T, 3, H, W)
T, H, W = input_scene.shape[1], input_scene.shape[3], input_scene.shape[4]

writer = imageio.get_writer(FLOW_OUTPUT, fps=fps, codec="libx264", macro_block_size=None)
compute_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

with torch.inference_mode():
    for start in range(0, T - 1, FLOW_WINDOW_SIZE - 1):
        end   = min(start + FLOW_WINDOW_SIZE, T)
        chunk = input_scene[:, start:end].to(device)

        with torch.autocast(device_type=device, dtype=compute_dtype, enabled=(device == "cuda")):
            flow_pr = flow_model(chunk, num_reg_refine=FLOW_ITERS)["flow_preds"][-1]

        if FLOW_RESTORE and native_size is not None:
            scaled = F.interpolate(flow_pr.view(-1, 2, H, W), size=native_size, mode="bilinear", align_corners=True)
            scaled[:, 0] *= native_size[1] / W
            scaled[:, 1] *= native_size[0] / H
            flow_pr = scaled.view(*flow_pr.shape[:2], 2, *native_size)

        for flow in flow_pr[0].permute(0, 2, 3, 1).cpu().numpy():
            writer.append_data(flow_to_image(flow, convert_to_bgr=False))

writer.close()
print(f"Saved → {FLOW_OUTPUT}")

In [ ]:
# Display the flow video inline
display(Video(FLOW_OUTPUT, embed=True, width=720))

---
## 5. 🎯 Point Tracking

MegaFlow can track a dense grid of points across all frames.  
Point tracking is derived directly from the flow — no separate training is needed.

> *Note: visibility is not explicitly predicted, so points are tracked through occlusions.*

In [ ]:
from megaflow.utils.basic import gridcloud2d
from megaflow.utils.visualizer import Visualizer

print("Loading megaflow-track from HuggingFace (auto-cached after first download)...")
track_model = MegaFlow.from_pretrained("megaflow-track", device=device)
track_model.eval()
print("Model ready.")

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
TRACK_GRID_SIZE = 8      # sub-sampling rate (larger → fewer tracked points)
TRACK_ITERS     = 8      # refinement iterations
TRACK_RESTORE   = True   # restore tracks/frames to original resolution
TRACK_OUTPUT    = "output_tracking"
# ────────────────────────────────────────────────────────────────────────────

frames_np, native_size, fps = load_video(INPUT_VIDEO, mode="track")
print(f"Loaded {len(frames_np)} frames  |  processing size: {frames_np[0].shape[:2]}  |  native: {native_size}  |  fps: {fps:.1f}")

input_image = [torch.from_numpy(f).permute(2, 0, 1).float() for f in frames_np]
frames = torch.stack(input_image, dim=0)[None].to(device)  # (1, T, 3, H, W)
B, T, _, H, W = frames.shape

grid_xy = gridcloud2d(1, H, W, norm=False, device=device).float()
grid_xy = grid_xy.permute(0, 2, 1).reshape(1, 1, 2, H, W)

compute_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

with torch.inference_mode():
    with torch.autocast(device_type=device, dtype=compute_dtype, enabled=(device == "cuda")):
        flows_e   = track_model.forward_track(frames, num_reg_refine=TRACK_ITERS)["flow_final"]
        traj_maps = flows_e.to(device) + grid_xy

traj_sub    = traj_maps[..., ::TRACK_GRID_SIZE, ::TRACK_GRID_SIZE]
pred_tracks = traj_sub.flatten(3).permute(0, 1, 3, 2)  # (1, T, N, 2)

if TRACK_RESTORE and native_size is not None:
    orig_H, orig_W = native_size
    pred_tracks[..., 0] *= orig_W / W
    pred_tracks[..., 1] *= orig_H / H
    frames = F.interpolate(frames[0], size=(orig_H, orig_W), mode="bilinear", align_corners=True).unsqueeze(0)

os.makedirs(TRACK_OUTPUT, exist_ok=True)
video_name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
vis = Visualizer(save_dir=TRACK_OUTPUT, pad_value=0, linewidth=1, tracks_leave_trace=0, fps=fps)
vis.visualize(frames, pred_tracks, filename=video_name, opacity=0.5)
print(f"Saved → {TRACK_OUTPUT}/{video_name}.mp4")

In [ ]:
# Display the tracking video inline
video_name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
display(Video(f"{TRACK_OUTPUT}/{video_name}.mp4", embed=True, width=720))